# Playground Series S6E9 (Predicting Electric Vehicle Purchases) / 「S6E9 LightGBM」解説付き写し

| 項目 | 内容 |
|---|---|
| コンペ | [Playground Series S6E9 — Predicting Electric Vehicle Purchases](https://www.kaggle.com/competitions/playground-series-s6e9) |
| 元notebook | [S6E9 LightGBM](https://www.kaggle.com/code/kirill0212/s6e9-lightgbm) |
| 原著者 | cstdy (kirill0212) |
| スコア | Public LB **0.94607**（公開notebookの最上位帯）/ 34 votes / Silver |
| ライセンス | Apache 2.0 |
| 日付 | 2026-09-03 |

> **この notebook について**
> 学習目的の「解説付き写し」です。原著のコードセルはそのまま保持し、各コードセルの直前に日本語の解説Markdownセルを挿入しています。未実行なので出力は含まれません。

---

## 手法の概要（1段落）

「その人が電気自動車を買うか（`Will_Buy_EV`）」を13個の属性から予測する二値分類。タイトルは LightGBM ですが、中身は**1本のパイプラインに6種類のモデル（LightGBM / XGBoost / CatBoost / TabM / RealMLP / MASA-FT-Transformer）を差し替えられるスイッチボード**になっており、`model_type = 'lgb'` の設定で LB 0.94607 を出しています。スコアを支えているのはモデルではなく特徴量の作り方で、柱は3つ。(1) **桁分解特徴量（digit features）**——各数値列を10のべき乗で割った余りを取り、`Annual_Income_USD` の1の位・10の位…をそれぞれ独立した特徴として与える。(2) **頻度エンコーディング**——各カテゴリ値の出現割合を train+test 全体から計算して与える（ラベルを使わないので統合しても安全）。(3) **入れ子のターゲットエンコーディング**——外側5-fold の学習側だけを見て、さらに内側5-fold の交差適合でエンコードする二重CV構造。加えて、定数列と完全相関列を機械的に落とし、OOF（out-of-fold）と pooled OOF の両方でAUCを報告します。

---

## 評価指標（Metric）

**指標は ROC-AUC**（`roc_auc_score`）。予測確率をしきい値で切らず、**「正例に付けたスコア＞負例に付けたスコア」となるペアの割合**をそのまま測ります。0.5 がランダム、1.0 が完全分離。

**なぜAUCなのか**：Playground の二値分類でAUCが選ばれるのは主に2つの理由です。

1. **しきい値を選ばなくてよい**。Accuracy や F1 は「0.5で切る」といった任意の決定が入り、その決め方だけでスコアが動いてしまう。AUCは順位だけを見るので、モデルの識別能力そのものを測れる。
2. **クラス不均衡に対して安定**。正例率が偏っていても、AUCは正例と負例をそれぞれ独立に扱うので、多数派に全部寄せる戦略が得をしない。

**この設計への影響（ここが実務的に重要）**：AUCは**順位不変（rank-invariant）**です。予測値を単調増加な関数（例えばシグモイド、対数、パーセンタイル順位への変換）で写しても、AUCは1ビットも変わりません。したがって——

- **確率のキャリブレーション（校正）はAUCを1ミリも改善しない。** 「予測を0〜1に整えました」という改良は、AUCコンペでは原理的に無意味です。
- **有効なのは順位を入れ替える操作だけ。** 新しい特徴量、別アーキテクチャ、そして**モデル間のブレンド**（異なる順位付けを混ぜるので順位が変わる）。
- 逆に言うと、**ランク平均のアンサンブルはAUCと非常に相性が良い**（元の予測値のスケールを気にせず混ぜられる）。今回のS6E9上位帯にランクブレンド系が並ぶのはこのためです。

**このnotebookが指標に対してどう設計されているか**：

- `eval_metric='auc'` と `early_stopping(500)` で、**AUCそのものを見て学習を止める**。Logloss で止めると、AUCの最適点とズレることがあります。
- fold ごとのAUCの平均（`mean AUC`）と、OOF予測をプールしてから1回で測ったAUC（`pooled OOF AUC`）の**両方**を印刷する。この2つが乖離するときは fold 間で予測のスケールが揃っていない（＝ブレンドで壊れやすい）サインです。
- ターゲットエンコーディングを**外側foldの内側でさらにCVして**作ることで、OOF AUC が楽観バイアスを持たないようにしている。ここが崩れると「手元0.96・LB 0.93」という典型的な事故になります。

---

## 学習ポイント（Playground系の合成データ特有の話）

Playground Series のデータは**実データから学習した生成モデルで合成**されています。その副作用として、値が特定の格子や丸め幅に乗っていることが多く、**桁を分解すると「生成器の癖」が特徴量として露出する**ことがあります。このnotebookの `FE()` はまさにそれを狙ったもので、実データではまず効かない一方、Playgroundでは定番の一手になっています。

「なぜ効くのか」を理解せずにコピーすると、実務データで同じことをして次元だけ爆発させることになるので、**合成データ限定のテクニックだと認識しておくこと**が大事です。


### 【設定】ライブラリ読み込みとハイパーパラメータ・特徴量スイッチ

**何をしているか**：必要なライブラリを読み込み、6種類のモデルそれぞれのハイパーパラメータ辞書（`params_xgb` / `params_cat` / `params_lgb` など）と、**特徴量生成のON/OFFスイッチ**（`USE_ORD`, `USE_FREQ`, `USE_ENC`, `USE_DIGIT`, `USE_PAIRS`, …）をまとめて定義します。最後の `model_type = 'lgb'` が、このバージョンで実際に使うモデルの選択です。

**なぜそうするのか**：Playground のような短期コンペでは「特徴量セット × モデル」の組み合わせを何十通りも試します。毎回コードを書き換えるとどれを試したか分からなくなるので、**フラグを並べた1つのセルを実験の操作卓にする**わけです。バージョン間の差分がこのセルだけになるので、後から履歴を読み返せます。

**LightGBM のパラメータを読む**：

| パラメータ | 値 | 意味 |
|---|---|---|
| `n_estimators` | 20000 | 木の本数の上限。実際は early stopping で止まる |
| `learning_rate` | 0.02 | 1本の木が結果に与える影響。小さい＝ゆっくり慎重に学ぶ |
| `max_depth` / `num_leaves` | 5 / 247 | 木の深さの上限と葉の数の上限。LightGBMは葉単位で成長するので `num_leaves` が主役 |
| `colsample_bytree` | **0.303** | 木1本あたりが使う特徴量の割合。**3割だけ**しか見せない |
| `subsample` | 0.813 | 木1本あたりが使う行の割合 |
| `min_child_samples` | 10 | 葉に最低10サンプル。少なすぎる葉＝ノイズを覚えた葉を作らせない |
| `reg_alpha` / `reg_lambda` | 0.07 / 2.03 | L1 / L2 正則化 |
| `max_bin` | 1024 | 連続値を何段階に離散化するか。既定255より細かく、桁特徴量の微細な差を潰さない |

> 💡 **`colsample_bytree` を 0.3 まで下げる理由**：この後の特徴量生成で列数が元の13列から数百列に膨らみます（桁分解＋頻度＋ターゲットエンコーディング）。似た情報を持つ列が大量にあるとき、毎回全部見せると**どの木も同じ強い列ばかり使い、木同士が似てしまう**。列をランダムに絞ると木の多様性が上がり、アンサンブルとしての性能が伸びます。列数が多いほど小さくするのがセオリーです。

In [ ]:
from pathlib import Path
import time
from functools import reduce
import math
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, KBinsDiscretizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
import os
from tqdm import tqdm
from itertools import combinations
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except:
    pass
import warnings
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMRegressor,LGBMClassifier,log_evaluation,early_stopping
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
pd.set_option('display.max_columns', 50)

TARGET = 'Will_Buy_EV'
ID_COL = 'id'
SEED = 42
N_FOLDS = 5

params_xgb = dict(
    n_estimators=12_000,
    learning_rate=0.012,
    max_depth=6,
    colsample_bytree=0.3,
    subsample=0.8,
    min_child_weight=10.0,
    # reg_lambda=5.0,
    max_bin=1024,
)

params_cat = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 15000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'random_seed': 42,
    'early_stopping_rounds': 1000,
    'rsm': 0.3,
    # 'task_type': 'GPU',
    # 'devices': '0'
}

params_lgb = {
    'random_state': 60,
    'feature_pre_filter': False,
    'verbose': -1,
    'n_estimators': 20000,
    'learning_rate': 0.02,
    'max_depth': 5,
    'min_child_samples': 10,
    'subsample': 0.812763123433567,
    'colsample_bytree': 0.3029300829885024, 
    'num_leaves': 247, 
    'reg_alpha': 0.07094285437903122,
    'reg_lambda': 2.033039097703242495,
    'max_bin': 1024,
}

RAW_CAT = ['Annual_Income_USD', 'Daily_Commute_km', 'Age', 'Charging_Stations_Near_Home', 
           'Charging_Stations_Near_Work']
RAW_CAT_100 = ['Annual_Income_USD', 'Daily_Commute_km']
RAW_CAT_500 = ['Annual_Income_USD', 'Daily_Commute_km']
NUMS = ['Age', 'Annual_Income_USD', 'Daily_Commute_km',
       'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
       'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender',
       'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level']

USE_ORD = False
USE_FREQ = True
USE_ENC = True
USE_DIGIT = True
USE_BASE = False
USE_CATEGORY = False
USE_PAIRS = False
USE_BIN_CATEGORY = False
USE_BIN_TE = False
USE_BIN_10 = False
USE_BIN_100 = False
USE_BIN_500 = False

model_type = 'lgb'

### 【前処理】カテゴリ変数の数値化

**何をしているか**：`Gender`, `City_Type`, `Current_Car_Type`, `Home_Charging_Possible`, `Subsidy_Available`, `Range_Anxiety_Level` の文字列を、辞書 `.map()` で整数に置き換えます。目的変数 `Will_Buy_EV` も `No→0 / Yes→1` に。

**なぜそうするのか**：勾配ブースティング木は数値しか受け取れません。ここでは **序数エンコーディング（ordinal encoding）** ——単純に整数を割り当てる方式を使っています。

`Range_Anxiety_Level` の `Low→0, Medium→1, High→2` は**本来の順序と一致している**ので理にかなっています。一方 `Current_Car_Type` の `Sedan→0, SUV→1, Hatchback→2, Truck→3` には順序の意味はありません。ただし**決定木は値の大小ではなく「どこで切るか」しか見ない**ので、木系モデルでは順序なしカテゴリに整数を振っても大きな害はありません（線形モデルやニューラルネットだと「SUVはSedanの1倍」といった誤った関係を学ぶので one-hot 等が必要）。

このセルの `t` は次のセルで `nunique()` を見るための下見用で、本番の読み込みは cell 4 で改めて行われます。

In [ ]:
try:
    t = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv')
except:
    t = pd.read_csv('train.csv')
def preprocess(t):
    t['Gender'] = t['Gender'].map({'Male': 0, 'Female': 1, 'Other': 2})
    t['City_Type'] = t['City_Type'].map({'Urban': 0, 'Suburban': 1, 'Rural': 2})
    t['Current_Car_Type'] = t['Current_Car_Type'].map({'Sedan': 0, 'SUV': 1, 'Hatchback': 2, 'Truck': 3})
    t['Home_Charging_Possible'] = t['Home_Charging_Possible'].map({'Yes': 0, 'No': 1})
    t['Subsidy_Available'] = t['Subsidy_Available'].map({'Yes': 0, 'No': 1})
    t['Range_Anxiety_Level'] = t['Range_Anxiety_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    if 'Will_Buy_EV' in t.columns:
        t['Will_Buy_EV'] = t['Will_Buy_EV'].map({'No': 0, 'Yes': 1})
    return t

### 【探索】各列のユニーク値数を確認

**何をしているか**：`t.nunique()` で列ごとの異なる値の個数を数えます。

**なぜそうするのか**：この1行が、以降の特徴量設計の**判断材料**になります。

- ユニーク数が極端に少ない列（2〜5）→ 実質カテゴリ。頻度エンコーディングやターゲットエンコーディングが効きやすい
- ユニーク数が非常に多い列（`Annual_Income_USD` など）→ 連続値。**桁分解**や**ビン化**の対象
- ユニーク数が1の列 → 情報ゼロ。落とす（cell 4 で自動的に落とされます）

EDA（探索的データ分析）を長大なグラフの束にせず、**次の意思決定に直結する1行だけを打つ**というのは、時間の限られたコンペでは合理的なスタイルです。

In [ ]:
t.nunique()

### 【特徴量】桁分解特徴量（digit features）— このnotebookの主武器

**何をしているか**：`FE()` の中身が全てです。

```python
for c in NUMS:
    for k in range(-4, 4):
        df[f"{c}_digit{k}"] = (df[c].fillna(0) // (10**k) % 10).astype('int8')
```

各数値列 `c` について、`k = -4..3` の8通りで「10^k で割った商の1の位」を取り出します。`k=0` なら1の位、`k=1` なら10の位、`k=-1` なら小数第1位、`k=-2` なら小数第2位……という具合です。13列 × 8桁 = **104列**が新たに生まれます。

**なぜそうするのか（重要）**：Playground Series のデータは実データを模した**合成データ**です。生成器は多くの場合、値を一定の丸め幅に乗せたり、特定の分布から離散的にサンプリングしたりします。その結果、**「小数第2位が0である」「10の位が7である」といった事実そのものが目的変数と相関する**ことがあります。人間の目には意味不明ですが、生成過程の指紋（fingerprint）が残っているわけです。

木モデルは `Annual_Income_USD > 52340` のような閾値分割はできますが、「1の位が3」のような**周期的・非単調な構造は表現が非常に苦手**です（何十回も分割を重ねる必要がある）。桁を列として明示的に与えることで、木は1回の分割でその情報を使えるようになります。

> ⚠️ **実データには基本効きません。** これは「合成データの生成器の癖を突く」テクニックです。効いた場合も、それは対象の現象を理解したからではなく、データの作られ方を当てたからです。Kaggleのスコアとしては正当ですが、実務にそのまま持ち込むと**列数だけ8倍になって過学習する**ので注意してください。

> 💡 `//` は切り捨て除算、`%` は剰余。`(1234 // 10**1) % 10` = `123 % 10` = `3`（10の位）。`int8` にキャストしているのはメモリ節約で、値が0〜9しか取らないので1バイトで足ります。

In [ ]:
def find_competition_dir() -> Path:
    candidates = [
        Path('/kaggle/input/competitions/playground-series-s6e9'),
        Path('.'),
    ]
    for path in candidates:
        if (path / 'train.csv').is_file() and (path / 'test.csv').is_file():
            return path
    raise FileNotFoundError('Attach the Playground Series S6E8 competition data.')

def FE(df): 
    for c in NUMS:
        for k in range(-4,4):
            df[f"{c}_digit{k}"] = (df[c].fillna(0) // (10**k) % 10).astype('int8')
    return df 

def make_base_features(x):
    return x

def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame[[c for c in frame.columns if c not in (ID_COL, TARGET)]].copy()
    if USE_DIGIT:
        data = FE(data)
    if USE_BASE:
        data = make_base_features(data)
    return data

### 【特徴量】列の掃除・頻度エンコーディング・fold分割

**何をしているか**：4つの処理がまとまっています。

**(1) データ読み込みと特徴量生成**：`add_features()` で桁特徴量を追加。

**(2) 無意味な列の自動除去**
```python
corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [c for c in upper_tri.columns if any(upper_tri[c] == 1.0)]
DROP = set(定数列) | to_drop | ...
```
相関行列の**上三角だけ**を見て（`np.triu` で下半分をNaNにする。そうしないと A と B、B と A の両方が引っかかり両方消してしまう）、相関が**ちょうど1.0**の列＝完全な重複を片方だけ落とします。あわせて `nunique() == 1` の定数列も落とす。桁分解で作った104列には「その桁が常に0」のような無意味な列が必ず混ざるので、この掃除が必要になります。

**(3) 頻度エンコーディング（frequency encoding）**
```python
frequency = pd.concat([train[col], test[col]]).value_counts(normalize=True)
X[f'{col}_freq'] = train[col].map(frequency)
```
各値が**全体の何割を占めるか**を新しい特徴にします。「珍しい値かどうか」という情報は、値そのものとは別の軸で効きます。

ここでコメントに書かれている通り、**この変換はラベル（y）を一切使わない**ので、train と test を結合して計算しても**リークになりません**。むしろ結合したほうが頻度の推定が安定します。テストの特徴量分布を使うことは transductive learning と呼ばれ、Kaggleでは標準的に許容されています。

**(4) fold分割**
```python
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y))
```
**層化（stratified）** ——各foldの正例率を全体と揃える分割。二値分類では必ずこちらを使います。ここで一度リスト化して固定しておくのは、この後のモデル比較で**全モデルが完全に同じ分割**を使うようにするためです。分割が違うと0.001程度のスコア差は簡単に出るので、比較が無意味になります。

> ⚠️ **リークの境界線**：train+test を混ぜてよいのは「yを使わない変換」だけです。頻度・ビン境界・標準化のパラメータはOK。**ターゲットエンコーディングは絶対にダメ**（次の cell 10 で、わざわざ fold の内側で作り直しているのはこのためです）。

In [ ]:
data_dir = find_competition_dir()
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
train = preprocess(train)
test = preprocess(test)
y = train[TARGET].to_numpy(np.int8)

raw_train = train.drop(columns=[ID_COL, TARGET]).copy()
raw_test = test.drop(columns=[ID_COL]).copy()
X = add_features(train)
X_test = add_features(test)

corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] == 1.0)]

DROP=set([c for c in X_test.columns if X_test[c].nunique()==1]).union(to_drop).union([c for c in X.columns if X[c].nunique()==1])
print(f"DROP:{DROP}")
X.drop(DROP,axis=1,inplace=True)
X_test.drop(DROP,axis=1,inplace=True)

# These transforms do not use labels, so their vocabularies may safely use train and test.
if USE_ORD:
    for column in RAW_CAT:
        categories = raw_train[column].fillna('__NA__').astype(str).unique()
        category_map = {value: index for index, value in enumerate(categories)}
        X[f'{column}_ord'] = raw_train[column].fillna('__NA__').astype(str).map(category_map).astype('float32')
        X_test[f'{column}_ord'] = raw_test[column].fillna('__NA__').astype(str).map(category_map).fillna(len(categories)).astype('float32')

if USE_FREQ:
    raw_train_strings = raw_train.astype('string').fillna('__NA__')
    raw_test_strings = raw_test.astype('string').fillna('__NA__')
    for column in raw_train.columns:
        frequency = pd.concat([raw_train_strings[column], raw_test_strings[column]], ignore_index=True).value_counts(normalize=True)
        X[f'{column}_freq'] = raw_train_strings[column].map(frequency).astype('float32')
        X_test[f'{column}_freq'] = raw_test_strings[column].map(frequency).astype('float32')

if USE_CATEGORY:
    for col in RAW_CAT:
        cat_name = f"{col}_cat_"
        codes, uniques = np.floor(pd.concat([X[col], X_test[col]], ignore_index=True)).factorize()
        code_map = {cat: i for i, cat in enumerate(uniques)}
    
        codes = np.floor(X[col]).map(code_map).astype('int32')
        X[cat_name] = codes
    
        codes_test = np.floor(X_test[col]).map(code_map).astype('int32')
        X_test[cat_name] = codes_test
        X[cat_name] = X[cat_name].astype('category')
        X_test[cat_name] = X_test[cat_name].astype('category')

if USE_BIN_CATEGORY or USE_BIN_TE:
    dicts = []
    if USE_BIN_10:
        dicts.append({col: [10] for col in RAW_CAT})
    if USE_BIN_100:
        dicts.append({col: [100] for col in RAW_CAT_100})
    if USE_BIN_500:
        dicts.append({col: [500] for col in RAW_CAT_500})
    bin_config = reduce(lambda x, y: x | y, dicts)
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                kb = KBinsDiscretizer(
                    n_bins=n_bins,
                    encode='ordinal',
                    strategy=strategy,
                    subsample=None
                )
                binned = kb.fit_transform(pd.concat([X[[col]], X_test[[col]]], ignore_index=True)).ravel().astype('int32')
                binned = kb.transform(X[[col]]).ravel().astype('int32')
                X[bin_name] = binned
                X[bin_name] = X[bin_name].astype('category')
    
                binned_test = kb.transform(X_test[[col]]).ravel().astype('int32')
                X_test[bin_name] = binned_test
                X_test[bin_name] = X_test[bin_name].astype('category')
                if USE_BIN_TE:
                    raw_train_strings[bin_name] = X[bin_name].astype('string')
                    raw_test_strings[bin_name] = X_test[bin_name].astype('string')
                    if USE_BIN_CATEGORY == False:
                        X = X.drop(columns=[bin_name])
                        X_test = X_test.drop(columns=[bin_name])

digit_columns = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                  'work_study_hours', 'sleep_hours', 'weekend_screen_time',
                  'notifications_per_day', 'app_opens_per_day', 'age']

if USE_PAIRS:
    columns = NUMS
    for r in [2]:
        for cols in tqdm(list(combinations(columns, r))):
            if len(set(cols).intersection(NUMS)) < 1:
                continue
            name = '-'.join(cols)
    
            raw_train_strings[name] = X[cols[0]].astype(str)
            for col in cols[1:]:
                raw_train_strings[name] = raw_train_strings[name] + '_' + X[col].astype(str)
    
            raw_test_strings[name] = X_test[cols[0]].astype(str)
            for col in cols[1:]:
                raw_test_strings[name] = raw_test_strings[name] + '_' + X_test[col].astype(str)
    
            combined = pd.concat([raw_train_strings[name], raw_test_strings[name]], ignore_index=True)
            combined, _ = combined.factorize()
            if pd.Series(combined).nunique() > len(combined) // 2:
                raw_train_strings = raw_train_strings.drop(name, axis=1)
                raw_test_strings = raw_test_strings.drop(name, axis=1)
                continue
            raw_train_strings[name] = combined[:len(raw_train_strings)]
            raw_test_strings[name] = combined[len(raw_train_strings):len(raw_train_strings) + len(raw_test_strings)]
            raw_train_strings[name] = raw_train_strings[name].astype('string')
            raw_test_strings[name] = raw_test_strings[name].astype('string')

cat_cols = [col for col in X.columns if col.endswith('_')]

folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X, y))
print(f'train={X.shape}, test={X_test.shape}, positive_rate={y.mean():.6f}')

### 【準備】追加ライブラリのインストール

**何をしているか**：`pytabkit`（TabM / RealMLP などのテーブルデータ用ニューラルネット実装）、`masamlp`（MLP系分類器）、`catstat`（高機能なターゲットエンコーダ）を pip で入れます。`%%capture` はセルの出力を丸ごと捨てるマジックコマンドで、インストールログを黙らせるためのものです。

**なぜそうするのか**：`model_type` を切り替えたときにこれらが必要になるからです。今回の実行（`lgb`）では `catstat` の `TargetEncoder` だけが実際に使われます。

> 💡 **Playground でニューラルネットを混ぜる価値**：GBDT（勾配ブースティング決定木）は特徴量を軸に平行な階段関数で近似します。一方MLPやTransformer系は滑らかな関数を作る。**間違え方が違う**ので、両者をブレンドするとAUCが伸びることが多い。単体性能ではGBDTが勝つことが多くても、アンサンブル要員として価値があります。

In [ ]:
%%capture
!pip install -q pytabkit
!pip install -q masamlp
!pip install -q catstat

### 【準備】乱数固定（seed_everything）

**何をしているか**：`random`、`numpy`、`torch`（CPU/GPU両方）、そして `PYTHONHASHSEED` 環境変数まで、乱数の種を一括で固定する関数を定義します。

**なぜそうするのか**：再現性のためです。ただし正確に言うと、目的は「同じ数字が出ること」自体ではなく、**「スコアが変わったのは自分が変えたもののせいだ」と言えるようにすること**です。乱数が動いていると、改善0.0003と乱数のブレ0.0005が区別できません。

> ⚠️ **注意**：GPUを使う場合、`seed_everything` だけでは完全な再現性は得られません。cuDNNの畳み込みアルゴリズム選択などが非決定的だからです。厳密にやるなら `torch.backends.cudnn.deterministic = True` も要りますが、速度と引き換えになります。

> 💡 seed固定でスコアが大きく動く（例：0.945 ↔ 0.943）なら、それは**モデルが不安定**というサイン。複数seedの平均を取るべきです。

In [ ]:
import random
import torch
import pytabkit
from pytabkit import TabM_D_Classifier
import masamlp
from masamlp import MasaClassifier
import catstat
from catstat import TargetEncoder as TargetEncoder2
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed=42)

### 【モデル定義】RealMLP 相当の実装（`model_type='realmlp'` 用）

**何をしているか**：`NumericalPreprocessor` と、それを使う分類器を自前で実装しています。今回の実行（`lgb`）では**使われません**が、読む価値があります。

**`NumericalPreprocessor` の中身**：
- `median_center`：平均ではなく**中央値**を引く
- `robust_scale`：標準偏差ではなく **IQR（四分位範囲 = 75%点 − 25%点）**で割る
- `smooth_clip` / `l2_normalize`：外れ値の滑らかな圧縮と正規化

**なぜ平均・標準偏差ではないのか**：`Annual_Income_USD` のような収入データは裾が長く、極端な高所得者が数人いるだけで平均も標準偏差も大きく引っ張られます。中央値とIQRは**外れ値に鈍感（robust）**なので、標準化後の値が安定します。GBDTは順位しか見ないので標準化不要ですが、ニューラルネットは入力のスケールに敏感なので、この違いがそのまま性能差になります。

さらにコードには `q_diff == 0`（IQRがゼロ＝値の半分以上が同じ値）の場合に **min-max の半分で代替する**という手当てが入っています。ゼロ除算対策であると同時に、「桁分解で作った、ほとんど定数の列」を安全に扱うための実装です。

**`CONFIG` に並ぶ学習テクニック**：

| 設定 | 意味 |
|---|---|
| `n_ens: 10` | 内部で10個のモデルを並列に持って平均する（ミニアンサンブル） |
| `lr_sched: "flat_cos"` | 前半35%は学習率一定、後半はコサインで減衰 |
| `ema_decay: 0.9979` | 重みの指数移動平均。学習終盤の振動を均す |
| `ls_eps: 0.04` | ラベルスムージング。正解を1.0でなく0.96として学習し、過信を防ぐ |
| `p_drop_sched: "expm4t"` | dropout率を学習の進行に応じて変化させる |
| `grad_clip: 1.2` | 勾配のノルムを1.2で頭打ちにして発散を防ぐ |

> 💡 **EMA（指数移動平均）**：`w_ema = 0.998 * w_ema + 0.002 * w_now` のように重みを平滑化して、それを推論に使う手法。ほぼタダで安定性が上がるので、近年の学習レシピではほぼ標準装備です。

In [ ]:
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]

    def fit(self, X, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5 * (X.max(axis=0)[zero_idx] - X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0 / (q_diff + 1e-30)
            self._iqr_factors[q_diff == 0.0] = 0.0
        return self

    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm == "median_center":
                X -= self._median[None, :]
            elif tfm == "robust_scale":
                X *= self._iqr_factors[None, :]
            elif tfm == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif tfm == "l2_normalize":
                norms = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(norms == 0, 1.0, norms)
        return X

class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8):
        super().__init__()
        self.n_ens = n_ens
        self.cat_dims = cat_dims
        self.onehot_features = []
        self.embed_layers = nn.ModuleList()
        self._embed_feature_indices = []
        for i, dim in enumerate(cat_dims):
            if dim <= onehot_thresh:
                self.onehot_features.append(i)
            else:
                emb = nn.ModuleList([nn.Embedding(dim, embed_dim) for _ in range(n_ens)])
                self.embed_layers.append(emb)
                self._embed_feature_indices.append(i)

    def forward(self, x):
        batch_size, n_ens, _ = x.shape
        features = []
        if self.onehot_features:
            onehot_x = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_oh = sum(onehot_dims)
            encoded = torch.zeros(batch_size, n_ens, total_oh, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx:idx+1].long()
                encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(encoded)
        for emb_list, feat_idx in zip(self.embed_layers, self._embed_feature_indices):
            feat_embs = []
            for model_idx in range(self.n_ens):
                indices = x[:, model_idx, feat_idx:feat_idx+1].long()
                feat_embs.append(emb_list[model_idx](indices))
            feat_combined = torch.cat(feat_embs, dim=1)
            features.append(feat_combined)
        return torch.cat(features, dim=2)

class ScalingLayer(nn.Module):
    def __init__(self, n_ens, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]

class NTPLinear(nn.Module):
    def __init__(self, n_ens, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        x = torch.einsum("bki,kio->bko", x, self.weight) / math.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x

class PBLDEmbedding(nn.Module):
    def __init__(self, n_ens, n_features, hidden_dim=16, out_dim=4, freq_scale=0.1, activation=nn.GELU):
        super().__init__()
        self.n_ens = n_ens
        self.n_features = n_features
        self.out_dim = out_dim
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) / math.sqrt(hidden_dim))
        self.b2 = nn.Parameter(torch.zeros(n_ens, n_features, out_dim - 1))
        self.act = activation()
        nn.init.uniform_(self.b1, -math.pi, math.pi)

    def forward(self, x):
        periodic = torch.cos(2 * math.pi * (x.unsqueeze(-1) * self.w1.unsqueeze(0) + self.b1.unsqueeze(0)))
        transformed = self.act(torch.einsum("bkfh,kfhd->bkfd", periodic, self.w2) + self.b2.unsqueeze(0))
        feat = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        return feat.flatten(start_dim=2)

class RealMLP(nn.Module):
    def __init__(self, output_dim, cat_dims, n_numerical, cfg):
        super().__init__()
        n_ens = cfg["n_ens"]
        embed_dim = cfg["embed_dim"]
        self.n_ens = n_ens
        self.cate = CategoricalFeatureLayer(n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim, onehot_thresh=cfg["onehot_thresh"])
        self.num_embed = PBLDEmbedding(n_ens=n_ens, n_features=n_numerical, hidden_dim=cfg["pbld_hidden_dim"],
                                        out_dim=cfg["pbld_out_dim"], freq_scale=cfg["pbld_freq_scale"], activation=cfg["pbld_activation"])
        num_emb_dim = n_numerical * cfg["pbld_out_dim"]
        cat_emb_dim = sum(c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total_dim = num_emb_dim + cat_emb_dim
        act = cfg["activation"]
        layers = []
        if cfg["add_front_scale"]:
            layers.append(ScalingLayer(n_ens=n_ens, n_features=total_dim))
        self._dropout_modules = []
        in_dim = total_dim
        for i, out_dim_h in enumerate(cfg["hidden_dims"]):
            linear = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=out_dim_h)
            if i == 0:
                self.first_linear = linear
            drop = nn.Dropout(cfg["dropout"])
            self._dropout_modules.append(drop)
            layers += [linear, act(), drop]
            in_dim = out_dim_h
        self.hidden = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=output_dim)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        x = self.hidden(torch.cat([x_num, x_cat], dim=2))
        return F.softmax(self.output_layer(x), dim=2)

def apply_schedule(init_value, progress, sched, flat_ratio=0.3):
    if sched == "constant":
        return init_value
    elif sched == "cos":
        return init_value * (math.cos(math.pi * progress) + 1) / 2
    elif sched == "flat_cos":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (math.cos(math.pi * t) + 1) / 2
    elif sched == "flat_anneal":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (1 - t)
    elif sched == "sqrt_cos":
        return init_value * math.sqrt((math.cos(math.pi * progress) + 1) / 2)
    elif sched == "expm4t":
        return init_value * math.exp(-4 * progress)
    else:
        raise ValueError(f"Unknown schedule: '{sched}'")

def get_parameter_groups(model, p):
    first_linear_weight_id = id(model.first_linear.weight)
    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if "num_embed" in name:
            pbld_p.append(param)
        elif "scale" in name:
            scale_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif "bias" in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)
    LR = p["lr"]
    WD = p["weight_decay"]
    return [
        {"params": scale_p, "lr": LR * p["lr_scale_mult"], "weight_decay": WD * p["wd_scale_mult"], "group": "scale"},
        {"params": pbld_p, "lr": LR * p["pbld_lr_factor"], "weight_decay": WD, "group": "pbld"},
        {"params": first_w_p, "lr": LR * p["first_layer_lr_factor"], "weight_decay": WD * p["first_layer_wd_factor"], "group": "first_w"},
        {"params": other_w_p, "lr": LR, "weight_decay": WD, "group": "other_w"},
        {"params": bias_p, "lr": LR * p["lr_bias_mult"], "weight_decay": WD * p["wd_bias_mult"], "group": "bias"},
    ]

def smooth_ce_loss(y_true, y_pred, ls=0.0, class_weights=None):
    n_classes = y_pred.size(1)
    y_smooth = torch.full_like(y_pred, ls / n_classes)
    y_smooth.scatter_(1, y_true.unsqueeze(1), 1.0 - ls + ls / n_classes)
    per_sample_loss = -(y_smooth * torch.log(y_pred.clamp(1e-15, 1))).sum(dim=1)
    if class_weights is not None:
        sample_weights = class_weights[y_true]
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum()
    return per_sample_loss.mean()

class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self, **kwargs):
        self.params = {**CONFIG, **kwargs}

    def fit(self, X_train, y_train, X_val, y_val, cat_col_names=None, X_test=None):
        p = self.params
        dev = torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        verbose = p["verbosity"]
        cat_col_names = cat_col_names or []
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]

        X_tr_num = X_train[num_col_names].values.astype(np.float32)
        X_val_num = X_val[num_col_names].values.astype(np.float32)
        X_tr_cat = X_train[cat_col_names].values.astype(np.int64)
        X_val_cat = X_val[cat_col_names].values.astype(np.int64)
        y_tr = np.asarray(y_train)
        y_v = np.asarray(y_val)

        self.preprocessor_ = NumericalPreprocessor(p["tfms"])
        self.preprocessor_.fit(X_tr_num)
        X_tr_num = self.preprocessor_.transform(X_tr_num)
        X_val_num = self.preprocessor_.transform(X_val_num)

        self.cat_col_names_ = cat_col_names
        self.num_col_names_ = num_col_names
        if cat_col_names:
            X_tr_cat = X_tr_cat + 1
            X_val_cat = X_val_cat + 1
            X_tr_cat = np.where(X_tr_cat < 1, 0, X_tr_cat)
            X_val_cat = np.where(X_val_cat < 1, 0, X_val_cat)
            all_cat = [X_tr_cat, X_val_cat]
            if X_test is not None:
                X_test_cat = X_test[cat_col_names].values.astype(np.int64) + 1
                X_test_cat = np.where(X_test_cat < 1, 0, X_test_cat)
                all_cat.append(X_test_cat)
            cat_dims = (np.concatenate(all_cat, axis=0).max(axis=0) + 1).tolist()
        else:
            cat_dims = []
        self.cat_dims_ = cat_dims

        if cat_dims:
            cat_max = np.array(cat_dims) - 1
            X_tr_cat = np.clip(X_tr_cat, 0, cat_max)
            X_val_cat = np.clip(X_val_cat, 0, cat_max)

        classes = np.unique(y_tr)
        self.classes_ = classes
        weights_np = compute_class_weight(class_weight=None, classes=classes, y=y_tr)
        class_weights = torch.as_tensor(weights_np, dtype=torch.float32, device=dev)
        n_classes = len(classes)

        self.model_ = RealMLP(output_dim=n_classes, cat_dims=cat_dims, n_numerical=X_tr_num.shape[1], cfg=p).to(dev)

        param_groups = get_parameter_groups(self.model_, p)
        for g in param_groups:
            g["lr_base"] = g["lr"]
        optimizer = torch.optim.AdamW(param_groups, betas=(p["mom"], p["sq_mom"]))

        Xtn = torch.as_tensor(X_tr_num, dtype=torch.float32, device=dev)
        Xtc = torch.as_tensor(X_tr_cat, dtype=torch.long, device=dev)
        ytt = torch.as_tensor(y_tr, dtype=torch.long, device=dev)
        Xvn = torch.as_tensor(X_val_num, dtype=torch.float32, device=dev)
        Xvc = torch.as_tensor(X_val_cat, dtype=torch.long, device=dev)

        n_ens = p["n_ens"]
        train_bs = p["train_bs"]
        eval_bs = p["eval_bs"]
        epochs = p["epochs"]
        lr_sched = p["lr_sched"]
        flat_ratio = p["flat_ratio"]
        ema_decay = p["ema_decay"]
        total_steps = epochs * len(y_tr)
        train_order = np.arange(len(y_tr))

        best_score = -np.inf
        best_epoch = 0
        best_val_probs = None
        best_state = None
        ema_state = None
        if ema_decay > 0:
            ema_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}

        for epoch in range(epochs):
            self.model_.train()
            for start in range(0, len(y_tr), train_bs):
                progress = (epoch * len(y_tr) + start) / total_steps
                idx_batch = train_order[start:start + train_bs]
                for g in optimizer.param_groups:
                    g["lr"] = apply_schedule(g["lr_base"], progress, lr_sched, flat_ratio)
                optimizer.zero_grad()
                y_pred = self.model_(Xtn[idx_batch], Xtc[idx_batch])
                ls_val = apply_schedule(p["ls_eps"], progress, p["ls_eps_sched"], flat_ratio)
                drop_val = apply_schedule(p["dropout"], progress, p["p_drop_sched"], flat_ratio)
                for dm in self.model_._dropout_modules:
                    dm.p = drop_val
                loss = smooth_ce_loss(
                    ytt[idx_batch].repeat_interleave(n_ens),
                    y_pred.reshape(-1, n_classes),
                    ls=ls_val, class_weights=class_weights)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), p["grad_clip"])
                optimizer.step()
                if ema_state is not None:
                    with torch.no_grad():
                        for key, value in self.model_.state_dict().items():
                            if torch.is_floating_point(value):
                                ema_state[key].mul_(ema_decay).add_(value.detach(), alpha=1.0 - ema_decay)
                            else:
                                ema_state[key].copy_(value)
            np.random.shuffle(train_order)

            self.model_.eval()
            if ema_state is not None:
                self.model_.load_state_dict(ema_state, strict=True)
            with torch.no_grad():
                val_probs = np.concatenate([
                    self.model_(Xvn[s:s + eval_bs], Xvc[s:s + eval_bs]).mean(dim=1).cpu().numpy()
                    for s in range(0, len(y_v), eval_bs)
                ], axis=0)
            epoch_score = roc_auc_score(y_v, val_probs[:, 1])
            improved = epoch_score > best_score
            if improved:
                best_score = epoch_score
                best_epoch = epoch + 1
                best_val_probs = val_probs.copy()
                state_src = ema_state if ema_state is not None else self.model_.state_dict()
                best_state = {k: v.detach().clone() for k, v in state_src.items()}
            if verbose >= 2:
                print(f"  epoch {epoch + 1}/{epochs}  auc = {epoch_score:.5f}  best = {best_score:.5f}  ls = {ls_val:.4f}  drop = {drop_val:.4f}" + (" v" if improved else ""))
            if p["use_early_stopping"]:
                patience = (best_epoch * p["early_stopping_multiplicative_patience"] + p["early_stopping_additive_patience"])
                if (epoch + 1) > patience:
                    if verbose >= 1:
                        print(f"  Early stopping at epoch {epoch + 1} (best epoch {best_epoch})")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state, strict=True)
        self.best_score_ = best_score
        self.best_val_probs_ = best_val_probs
        self._dev = dev
        if verbose >= 1:
            print(f"  best AUC: {best_score:.5f}  (epoch {best_epoch})")
        return self

    def predict_proba(self, X):
        eval_bs = self.params["eval_bs"]
        X_num = self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        X_cat = X[self.cat_col_names_].values.astype(np.int64) + 1
        X_cat = np.where(X_cat < 1, 0, X_cat)
        X_cat = np.clip(X_cat, 0, np.array(self.cat_dims_) - 1)
        Xn = torch.as_tensor(X_num, dtype=torch.float32, device=self._dev)
        Xc = torch.as_tensor(X_cat, dtype=torch.long, device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([
                self.model_(Xn[s:s + eval_bs], Xc[s:s + eval_bs]).mean(dim=1).cpu().numpy()
                for s in range(0, len(X_num), eval_bs)
            ], axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

CONFIG = {
    "n_ens": 10,
    "embed_dim": 16,
    "onehot_thresh": 8,
    "hidden_dims": [512, 512, 512],
    "dropout": 0.06,
    "p_drop_sched": "expm4t",
    "activation": nn.GELU,
    "add_front_scale": True,
    "pbld_hidden_dim": 20,
    "pbld_out_dim": 5,
    "pbld_freq_scale": 5.0,
    "pbld_activation": nn.PReLU,
    "pbld_lr_factor": 0.093,
    "lr": 0.01,
    "mom": 0.9,
    "sq_mom": 0.98,
    "lr_sched": "flat_cos",
    "flat_ratio": 0.35,
    "first_layer_lr_factor": 1.0,
    "first_layer_wd_factor": 0.1,
    "lr_scale_mult": 10.0,
    "lr_bias_mult": 0.1,
    "weight_decay": 0.013,
    "wd_scale_mult": 0.1,
    "wd_bias_mult": 0.5,
    "ema_decay": 0.997875,
    "grad_clip": 1.2,
    "ls_eps": 0.04,
    "ls_eps_sched": "cos",
    "tfms": ["median_center", "robust_scale"],
    "epochs": 4,
    "train_bs": 256,
    "eval_bs": 10240,
    "verbosity": 2,
    "use_early_stopping": True,
    "early_stopping_additive_patience": 1,
    "early_stopping_multiplicative_patience": 1,
    "device": "cuda",
    "random_state": 63,
}

### 【モデル定義】TabM のハイパーパラメータ（`model_type='tabm'` 用）

**何をしているか**：TabM 分類器の設定辞書。今回は未使用です。

**TabM とは**：テーブルデータ向けの比較的新しいニューラルネットで、**1つのMLPの中に k 個の「並行した頭」を持たせ、それらの平均を出力する**という構造です（`tabm_k: 32` が頭の数）。32個のモデルを別々に学習してアンサンブルするのと似た効果を、**重みの大部分を共有することで1回の学習コストで**得られます。

**注目すべき設定**：
- `val_metric_name: '1-auc_ovr'` — **AUCを直接見て早期終了する**。指標がAUCなのだから検証もAUCで、という一貫性
- `num_emb_type: 'pwl'` — 数値特徴を **piecewise-linear（区分線形）埋め込み**でベクトル化する。連続値をそのまま入れるより、ニューラルネットが非線形な閾値効果を学びやすくなる。GBDTが得意な「ある値で切る」挙動をNN側に持ち込むための工夫
- `n_epochs: 3` — 極端に短い。テーブルデータでは数エポックで十分（すぐ過学習する）ことの表れ

In [ ]:
params_tabm = {
    'device': 'cuda',
    'val_metric_name': '1-auc_ovr',
    'random_state': 42,
    'verbosity': 2,
    'tabm_k': 32,
    'num_emb_type': 'pwl',
    'd_embedding': 20,
    'batch_size': 64,
    'lr': 2e-3,
    'weight_decay': 2e-2,
    'n_epochs': 3,
    'dropout': 0.1,
    'd_block': 256,
    'n_blocks': 4
}

### 【モデル定義】FT-Transformer のハイパーパラメータ（`model_type='masa_ft'` 用）

**何をしているか**：FT-Transformer 系モデルの設定。今回は未使用です。

**FT-Transformer とは**：`Feature Tokenizer + Transformer`。各特徴量（列）を1つの「トークン」としてベクトル化し、Transformer の自己注意（self-attention）で**列同士の相互作用を学習**します。自然言語のTransformerが単語間の関係を学ぶのと同じ仕組みを、表の列に適用したものです。

**注目すべき設定**：
- `num_embedding: 'plr-lite'` — PLR（Periodic-Linear-ReLU）埋め込み。数値をsin/cosの周期関数に通してから線形層に入れる方式で、**細かい数値の差を分離しやすくなる**。桁分解特徴量と発想が近い（周期構造を明示的に与える）
- `lr_scheduler: 'cosine'` + `optimizer: 'adamw'` — 現代的な標準構成
- `n_epochs: 16` — TabM の3より長いのは、Transformer のほうが収束が遅いため

> 💡 **なぜGBDTと混ぜると効くのか**：GBDTは「ある列がある閾値を超えたか」の組み合わせで判断します。Transformer は「列Aと列Bの値の組み合わせ」を注意機構で直接見ます。**同じデータの違う側面を捉える**ので、予測の誤りが相関しにくく、ブレンドの利得が大きくなります。

In [ ]:
MODEL_PARAMS = dict(
    model='ft_transformer',
    n_epochs=16,
    batch_size=4096,
    learning_rate=0.001,
    weight_decay=1e-05,
    optimizer='adamw',
    optimizer_betas=None,
    lr_scheduler='cosine',
    weight_decay_schedule='none',
    grad_clip=None,
    num_embedding='plr-lite',
    numeric_scaler='quantile',
    cat_encoding='embedding',
    class_weight=None,
    label_smoothing=0.0,
    early_stopping_rounds=None,
    eval_metric='auc',
    n_ens=4,
    device='auto',
    amp='auto',
    verbose=2,
    ens_mode='loop',
    eval_batch_size=2048,
    model_params={'d_block': 128, 'n_blocks': 2, 'attention_n_heads': 8, 'n_frequencies': 24, 'sigma': 0.1},
)

### 【学習】外側5-fold × 入れ子ターゲットエンコーディング（このnotebookの2つめの主武器）

**何をしているか**：この notebook で最も重要なセルです。5-fold のループの中で、**foldごとにターゲットエンコーダを作り直しています**。

```python
for fold, (fit_index, valid_index) in enumerate(folds):
    encoder = TargetEncoder2(cv=5, smooth=smooth, stats=['mean'],
                             shuffle=True, random_state=seed, target_type='binary')
    encoded_fit   = encoder.fit_transform(raw_train_strings.iloc[fit_index], y[fit_index])
    encoded_valid = encoder.transform(raw_train_strings.iloc[valid_index])
    encoded_test  = encoder.transform(raw_test_strings)
```

**ターゲットエンコーディング（target encoding）とは**：カテゴリ値を「その値を取る行の目的変数の平均」に置き換える手法です。例えば `City_Type = Urban` の人の EV 購入率が 0.42 なら、`Urban → 0.42` とする。カテゴリの情報を1列に圧縮でき、高カーディナリティ（値の種類が多い）な列に特に強力です。

**しかし最もリークしやすい手法でもあります。** ある行のエンコード値を作るのに、その行自身の y が使われていたら、モデルは答えを見ながら学習することになります。手元のCVスコアだけが跳ね上がり、LBは伸びません。

**このnotebookの二重の防御**：

1. **外側**：エンコーダの `fit` に渡すのは `fit_index`（そのfoldの学習側）だけ。検証側 `valid_index` の y は絶対に見せない。だから OOF AUC が信用できる。
2. **内側**：`TargetEncoder2(cv=5)` により、学習側の中でもさらに5分割して**交差適合（cross-fitting）**する。行 i のエンコード値は、行 i を含まない4/5から計算される。これがないと、学習データ内部でだけ異常に良く効く列ができてしまい、モデルがそれに寄りかかって検証で崩れます。

**`smooth` のループ**：`for smooth in [10, 'auto']` と2種類作って**両方を特徴量として並べています**。smooth はスムージング（平滑化）の強さで、
```
エンコード値 = (そのカテゴリの合計 + smooth × 全体平均) / (そのカテゴリの件数 + smooth)
```
件数が少ないカテゴリほど全体平均に引き寄せられます。smooth が小さいと稀なカテゴリのノイズを拾い、大きいと情報が薄まる。**最適値を1つ選ぶのではなく、複数作ってモデルに選ばせる**——これは特徴量選択をモデルに委ねる実践的な戦略です（GBDTは有用でない列を無視するのが得意なので成立します）。

**学習と評価**：
```python
model.fit(x_fit, y[fit_index], eval_set=[(x_valid, y[valid_index])],
          eval_metric='auc', callbacks=[log_evaluation(500), early_stopping(500)])
oof[valid_index] = model.predict_proba(x_valid)[:, 1]
test_prediction += model.predict_proba(x_test)[:, 1] / N_FOLDS
```
- **AUCを見て500ラウンド改善なしなら停止**（`n_estimators=20000` は上限であって目標ではない）
- テスト予測は5つのfoldモデルの**単純平均**。これ自体が小さなアンサンブルです
- `oof` 配列に検証予測を書き溜めておく → 後でスタッキングやブレンド重み最適化の材料になる

**最後の2行が地味に重要**：
```python
print(f'mean AUC: {np.mean(fold_scores)}')     # fold ごとのAUCの平均
print(f'pooled OOF AUC: {roc_auc_score(y, oof)}')  # 全OOFをまとめて1回測る
```
この2つは**一致しません**。fold間で予測値のスケールが揃っていないと pooled のほうが下がります。AUCは順位しか見ないので、fold A の 0.6 と fold B の 0.6 が違う意味を持っていると、混ぜた瞬間に順位が壊れるのです。**両方を印刷して差を監視する**のは、ブレンド前に必ずやるべきチェックです。

In [ ]:
def numeric_frame(frame: pd.DataFrame) -> pd.DataFrame:
    return frame

started = time.time()
oof = np.zeros(len(train), dtype=np.float32)
test_prediction = np.zeros(len(test), dtype=np.float64)
fold_scores, best_iterations = [], []

for fold, (fit_index, valid_index) in enumerate(folds):
    # The encoder sees labels only from this outer fold's training partition.
    if USE_ENC:
        to_concat_fit = [numeric_frame(X.iloc[fit_index]).reset_index(drop=True)]
        to_concat_valid = [numeric_frame(X.iloc[valid_index]).reset_index(drop=True)]
        to_concat_test = [numeric_frame(X_test).reset_index(drop=True)]
        for seed in [42]:
            for smooth in [10, 'auto']:
                for inner_n_fold in [5]:
                    encoder = TargetEncoder2(
                        cv=5,
                        smooth=smooth,
                        stats=['mean'],
                        shuffle=True,
                        random_state=seed,
                        target_type='binary',
                    )
                    encoded_fit = encoder.fit_transform(raw_train_strings.iloc[fit_index], y[fit_index])
                    encoded_valid = encoder.transform(raw_train_strings.iloc[valid_index])
                    encoded_test = encoder.transform(raw_test_strings)
                    encoded_fit.columns = [col + f'_seed_{seed}_smooth_{smooth}_inner_n_fold_{inner_n_fold}' for col in encoded_fit.columns]
                    encoded_valid.columns = [col + f'_seed_{seed}_smooth_{smooth}_inner_n_fold_{inner_n_fold}' for col in encoded_valid.columns]
                    encoded_test.columns = [col + f'_seed_{seed}_smooth_{smooth}_inner_n_fold_{inner_n_fold}' for col in encoded_test.columns]
                    to_concat_fit.append(encoded_fit.reset_index(drop=True))
                    to_concat_valid.append(encoded_valid.reset_index(drop=True))
                    to_concat_test.append(encoded_test.reset_index(drop=True))
    
        x_fit = pd.concat(to_concat_fit, axis=1)
        x_valid = pd.concat(to_concat_valid, axis=1)
        x_test = pd.concat(to_concat_test, axis=1)
    else:
        x_fit = numeric_frame(X.iloc[fit_index]).reset_index(drop=True)
        x_valid = numeric_frame(X.iloc[valid_index]).reset_index(drop=True)
        x_test = numeric_frame(X_test).reset_index(drop=True)
    print(x_fit.shape)
    if model_type == 'xgb':
        model = xgb.XGBClassifier(
            objective='binary:logistic',
            tree_method='hist',
            device='cuda',
            eval_metric='auc',
            early_stopping_rounds=500,
            random_state=SEED + fold,
            **params_xgb,
        )
        model.fit(x_fit, y[fit_index], eval_set=[(x_valid, y[valid_index])], verbose=False)
    elif model_type == 'realmlp':
        model = RealMLP_TD_Classifier(**CONFIG)
        model.fit(x_fit, y[fit_index], x_valid, y[valid_index], cat_col_names=cat_cols)
    elif model_type == 'tabm':
        model = TabM_D_Classifier(**params_tabm)
        model.fit(x_fit, y[fit_index], x_valid, y[valid_index], cat_col_names=cat_cols)
    elif model_type == 'masa_ft':
        model = MasaClassifier(**MODEL_PARAMS, categorical_features=cat_cols, random_state=SEED)
        model.fit(x_fit, y[fit_index], eval_set=[(x_valid, y[valid_index])])
    elif model_type == 'cat':
        model = CatBoostClassifier(**params_cat)
    
        model.fit(
            x_fit, y[fit_index],
            eval_set=(x_valid, y[valid_index]),
            verbose=500,
            cat_features=cat_cols
        )
    elif model_type == 'lgb':
        model=LGBMClassifier(**params_lgb)

        model.fit(x_fit, y[fit_index], eval_set=[(x_valid, y[valid_index])],
                  eval_metric='auc',
                  callbacks=[log_evaluation(500),early_stopping(500)])

    valid_prediction = model.predict_proba(x_valid)[:, 1]
    oof[valid_index] = valid_prediction
    test_prediction += model.predict_proba(x_test)[:, 1] / N_FOLDS

    fold_auc = roc_auc_score(y[valid_index], valid_prediction)
    fold_scores.append(fold_auc)
    print(f'fold={fold} auc={fold_auc:.6f}')

print(f'mean AUC: {np.mean(fold_scores):.9f}')
pooled_auc = roc_auc_score(y, oof)
print(f'pooled OOF AUC: {pooled_auc:.9f}')
print(f'elapsed: {time.time() - started:.1f}s')

### 【出力】OOF・テスト予測・提出ファイルの保存

**何をしているか**：
```python
np.save(f'oof_predictions_{model_type}_{SEED}_{N_FOLDS}.npy', oof)
np.save(f'test_predictions_{model_type}_{SEED}_{N_FOLDS}.npy', test_prediction)
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_prediction})
submission.to_csv('submission.csv', index=False)
```

**なぜOOFとテスト予測を別ファイルで保存するのか**：これがアンサンブルの土台になるからです。ファイル名に `{model_type}_{SEED}_{N_FOLDS}` を埋め込んでいるので、`lgb` / `xgb` / `cat` / `tabm` と順に実行すれば、同じfold分割で作られた予測ペアが並びます。あとは別notebookで

1. 各モデルのOOFを使って**ブレンド重みを最適化**（OOF AUCが最大になる重みを探す）
2. その重みをテスト予測に適用して提出

とすれば、正しい手順のスタッキングになります。**同じfold分割を使っていることが前提条件**で、cell 4 で `folds` をリストとして固定しておいたのはこのためです。

> 💡 **提出値は確率のままでよい**：AUCは順位不変なので、0〜1に収まっていなくても、キャリブレーションされていなくても、順位さえ正しければスコアは同じです。逆にLoglossのコンペならここでキャリブレーションが必須になります。**指標を見てから後処理を決める**という順番を忘れないでください。

In [ ]:
np.save(f'oof_predictions_{model_type}_{SEED}_{N_FOLDS}.npy', oof)
np.save(f'test_predictions_{model_type}_{SEED}_{N_FOLDS}.npy', test_prediction.astype(np.float32))
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_prediction,
})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print(f'Wrote submission.csv with {len(submission):,} rows')

### 【末尾】空セル

原著notebookの末尾にある空のセルです。写しの忠実性を保つためそのまま残しています。